## Let's get the domains for the entire tree and plot them

In [32]:
from collections import defaultdict  # LP: required for parsing the domain data
from pathlib import Path

# We'll need ETE3 to parse and render the tree
from ete3 import Tree, TreeStyle, NodeStyle, TextFace, CircleFace, faces, SeqMotifFace
from tqdm.notebook import tqdm
import pandas as pd

In [6]:
# File locations and other constants
treepath = Path("TBE_named_middoint.new")  # transporter family tree, Newick format
annopath = Path("all_annotations.csv")     # annotations for some family members


In [7]:
# Load the tree
# We have these as functions so that we can annotate/render a clean tree each time
def load_tree():
    tree = Tree(str(treepath))
    return tree

# Load the annotation
def load_annotation():
    anno = pd.read_csv(annopath)
    return anno

# The function lets us return the tree and its corresponding annotation,
# where the leaves of this specific tree object are present in the
# annotation dataframe, and so can be manipulated easily.
def annotate_tree():
    tree = load_tree()
    anno = load_annotation()

    leaves = []  # Will hold leaves for the tree where they match the annotation row, or None if there is none
    
    for id in anno["0"]:               # iterate over annotations
        for leaf in tree.iter_leaves():  # iterate over all leaves in the tree
            assigned = False
            if id in str(leaf):
                leaves.append(leaf)
                assigned = True
                break
        if not assigned:
            leaves.append(None)
    
    anno["leaves"] = leaves

    return tree, anno

In [ ]:
tree, anno = annotate_tree()  # get a clean tree

# Declare a style (we could have put this in a function to save repeated code)
kingdom = TreeStyle()
kingdom.show_leaf_name = True
kingdom.mode = "c"
kingdom.show_branch_support = True

tree.ladderize()
R = tree.get_midpoint_outgroup()
tree.set_outgroup(R)

# One text face for each kingdom
face_dict = {"Eukaryota": TextFace("eukaryota"),
             "Bacteria": TextFace("bacteria"),
             "Archaea": TextFace("archaea")}

# Set colours for kingdoms
colour_dict = {"Eukaryota": "#FFFACD", "Bacteria": "#F0F8FF", "Archaea": "#FFE4E1"}

# Iterate over all annotated leaves and add the face
for idx, row in anno.iterrows():
    # Get annotation information
    leaf = row["leaves"]
    kingdomname = row["kingdom"].strip()


    # Set styling for the leaf node
    # leaf.img_style["size"] = 12
    leaf.img_style["bgcolor"] = colour_dict[kingdomname]    
    leaf.add_face(face_dict[kingdomname], 0, "aligned")

tree.render("figure_3.4.pdf", tree_style=kingdom, w=24, h=24, units="in");